# Importação de bibliotecas

In [1]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E CONFIGURAÇÕES GLOBAIS
# ==============================================================================
# 1. Bibliotecas de terceiros (manipulação de dados)
import pandas as pd

# 2. Bibliotecas nativas do Python
import warnings
import os

# Ignora avisos não críticos para manter o output limpo
warnings.filterwarnings("ignore")

# Configuração de caminhos e parametrização

In [2]:
# ==============================================================================
# 2. CONFIGURAÇÃO DE CAMINHOS (PATHS) E PARÂMETROS GERAIS
# ==============================================================================

# Entradas e Saídas (Inputs/Outputs)
BASE_PATH = '../data'
INPUT_DIR = f'{BASE_PATH}/raw/metrics'
OUTPUT_FILE = f'{BASE_PATH}/processed/dashboard_input/metrics_looker_studio.csv'

# Parâmetros de execução
YEARS = range(2016, 2026)

# ==============================================================================
# 3. DICIONÁRIO DE PADRONIZAÇÃO DE COLUNAS PARA O LOOKER STUDIO
# ==============================================================================

COLUMNS_MAPPING = {
    'Unnamed: 0': 'Sequência',
    'Unnamed: 1': 'Nome',
    'Posdoc': 'Posdoc concluídos',
    'Doutorado': 'Doutorados concluídos',
    'Mestrado': 'Mestrados concluídos',
    'Especialização': 'Especializações concluídas',
    'TCC': 'TCCs concluídos',
    'IC': 'ICs concluídas',
    'Posdoc.1': 'Posdoc em andamentos',
    'Doutorado.1': 'Doutorados em andamento',
    'Mestrado.1': 'Mestrados em andamento',
    'Especialização.1': 'Especializações em andamento',
    'TCC.1': 'TCCs em andamento',
    'IC.1': 'ICs em andamento'
}

# Extração e consolidação dos dados (Extract)

In [3]:
# ==============================================================================
# 4. EXTRAÇÃO DE TABELAS HTML E CONSOLIDAÇÃO DOS DADOS (EXTRACT)
# ==============================================================================
print(">>> Iniciando extração de tabelas HTML...")

dfs_list = []

# 1. Varredura e extração de tabelas por ano
for year in YEARS:
    file_path = f'{INPUT_DIR}/metrics_{year}.html'
    
    try:
        # A função read_html devolve uma lista de todas as tabelas na página web/arquivo
        html_tables = pd.read_html(file_path)

        # Pega a primeira tabela (Produções bibliográficas, técnicas e artísticas)
        df_productions = html_tables[0]
        
        # Pega a segunda tabela (Orientações concluídas e em andamento)
        # e guarda apenas da 8ª coluna em diante (ignora as colunas repetidas)
        df_advising = html_tables[1].iloc[:, 8:] 
        
        # Concatenação horizontal (axis=1) das tabelas de produção e orientação
        df_year = pd.concat([df_productions, df_advising], axis=1)
        df_year.insert(0, 'Ano', year)
        
        dfs_list.append(df_year)
        
    except FileNotFoundError:
        print(f"Aviso: O arquivo metrics_{year}.html não foi encontrado. Pulando...")
        
print(f"✅ Extração concluída. Total de anos processados: {len(dfs_list)}")

>>> Iniciando extração de tabelas HTML...
✅ Extração concluída. Total de anos processados: 10


# Limpeza e transformação (Transform)

In [4]:
# ==============================================================================
# 5. LIMPEZA, TRANSFORMAÇÃO E HIGIENIZAÇÃO DOS DADOS (TRANSFORM)
# ==============================================================================
print(">>> Aplicando transformações e higienização aos dados...")

# 1. Empilha todos os anos verticalmente
df_metrics = pd.concat(dfs_list, ignore_index=True)

# 2. Aplica as renomeações para o formato do dashboard
df_metrics.rename(columns=COLUMNS_MAPPING, inplace=True)

# 3. Formatação visual: converte a numeração para formato texto com ponto (ex: "1.")
df_metrics['Sequência'] = df_metrics['Sequência'].fillna(0).astype(int).astype(str) + '.'

# 4. Ordenação alfabética e cronológica (padrão de visualização)
df_metrics.sort_values(by=['Nome', 'Ano'], ascending=[True, True], inplace=True)

print("✅ Dados transformados e consolidados com sucesso!\n")
display(df_metrics.head(3))

>>> Aplicando transformações e higienização aos dados...
✅ Dados transformados e consolidados com sucesso!



,Ano,Sequência,Nome,Rótulo/Grupo,Bolsa CNPq,Período de análise individual,Data de atualização do CV,Grande área,Área,lat,...,Especializações concluídas,TCCs concluídos,ICs concluídas,Orientações em andamento,Posdoc em andamentos,Doutorados em andamento,Mestrados em andamento,Especializações em andamento,TCCs em andamento,ICs em andamento
58,2016,59.,Adriana Paes de Jesus Correia,Pedagogia,NaN,NaN,24/02/2021,Ciências Humanas,Educação,NaN,...,0,0,0,0,0,0,0,0,0,0
227,2017,59.,Adriana Paes de Jesus Correia,Pedagogia,NaN,NaN,24/02/2021,Ciências Humanas,Educação,NaN,...,0,0,0,0,0,0,0,0,0,0
396,2018,59.,Adriana Paes de Jesus Correia,Pedagogia,NaN,NaN,24/02/2021,Ciências Humanas,Educação,NaN,...,0,0,0,0,0,0,0,0,0,0


# Exportação para o Looker Studio (Load)

In [5]:
# ==============================================================================
# 6. EXPORTAÇÃO DA BASE DE DADOS PARA O LOOKER STUDIO (LOAD)
# ==============================================================================
print("\n>>> Exportando base de dados para o dashboard...")

# 1. Garante que a pasta de destino existe
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# 2. Exportação em utf-8-sig para garantir leitura de acentuação em todos os sistemas
df_metrics.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f"✅ Arquivo exportado com sucesso para: {OUTPUT_FILE}")


>>> Exportando base de dados para o dashboard...
✅ Arquivo exportado com sucesso para: ../data/processed/dashboard_input/metrics_looker_studio.csv
